# Urakatsu worker

Runs Urakatsu's detection, OCR and inpainting models on this notebook's GPU for the Urakatsu app on your computer.

1. Choose **Runtime, Run all**.
2. When the worker is up it prints a line that starts with `Connection: https://`. Copy that whole line.
3. In the app, open **Settings, Workers**, and paste it into your Colab worker's **Connection** field.

The worker opens a Cloudflare tunnel so the app can reach it. Your pages travel to this notebook through Cloudflare, which relays them and could read them. If the notebook restarts it gets a new address: paste the new line. Keep this tab open while you translate; the worker stops after 30 minutes without requests.

Models are saved in your Google Drive, in `Urakatsu/models`, so the next session does not download them again. Colab asks you to allow Drive access when the notebook starts. Models can take several GB, and when Drive is full the model fails to load. To skip Drive, untick **save_models_to_drive**: models are then downloaded every session.

This notebook asks for a T4 GPU. If Colab gives you none, choose **Runtime, Change runtime type, T4 GPU** and run it again. Colab's free tier limits how long and how often you can use a GPU, and Google may end sessions that serve other apps; see the [Colab FAQ](https://research.google.com/colaboratory/faq.html).

In [ ]:
#@title Start the worker
save_models_to_drive = True #@param {type:"boolean"}

import os
import shutil
import subprocess
import sys

if shutil.which("nvidia-smi") is None:
    print("No GPU on this runtime: models will run on the CPU, slowly. Choose Runtime, Change runtime type, T4 GPU.")

drive_mounted = False
if save_models_to_drive:
    from google.colab import drive

    try:
        drive.mount("/content/drive")
        os.environ["URAKATSU_MODELS"] = "/content/drive/MyDrive/Urakatsu/models"
        drive_mounted = True
        print("Models are saved in Google Drive, in Urakatsu/models.")
    except Exception as error:
        print(f"Google Drive is not connected ({error}). Models will download again next session.")

print("Installing Urakatsu (a few minutes the first time)...")
# Urakatsu does not use it, but transformers imports it when present: a build left behind by the
# torch upgrade stops every Transformers model from loading.
subprocess.check_call([sys.executable, "-m", "pip", "uninstall", "--yes", "--quiet", "torchaudio"])
subprocess.check_call([sys.executable, "-m", "pip", "install", "--quiet", "urakatsu[worker] @ git+https://github.com/karrot0/Urakatsu@main"])
print("Starting the worker. Paste the Connection line below into the app (Settings, Workers).")
!urakatsu worker --tunnel --idle-exit 30
if drive_mounted:
    drive.flush_and_unmount()  # finish writing to Drive before the runtime goes